<a href="https://colab.research.google.com/github/edmundzen/dengue-early-warning/blob/main/member4_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai

**MEMBER 4 : Gemini AI & Integration**

Integrate the Gemini API.
Generate AI recommendations/explanations based on the risk scores.
Help connect all the modules and test the complete workflow.
## Tasks Completed
- Connected to Google BigQuery.
- Read the `inspection_priority` table generated by Member 2.
- Integrated the Gemini API.
- Generated AI recommendations for high-risk dengue areas.
- Saved the recommendations as a CSV (`inspection_with_ai.csv`).
- Verified the output for dashboard integration.

## Workflow
BigQuery (inspection_priority)
⬇
Gemini API
⬇
AI Recommendations
⬇
CSV Output
⬇
Looker Studio Dashboard

## Step 1: Import Required Libraries

In [2]:
from google import genai
from google.genai import types
import pandas as pd
from google.cloud import bigquery
from google.colab import auth

## Step 2: Authenticate with Google Cloud

In [3]:
auth.authenticate_user()

## Step 3: Connect to BigQuery

In [4]:
PROJECT_ID = "dengue-early-warning"

client = bigquery.Client(project=PROJECT_ID)

print("Connected!")

Connected!


## Step 4: Load the `inspection_priority` Table

In [5]:
query = """
SELECT *
FROM `dengue-early-warning.dengue_ew.inspection_priority`
LIMIT 10
"""

inspection_df = client.query(query).to_dataframe()

inspection_df.head()

,rank,cell_id,lat,lon,date,risk_score,top_factor
0,276176,1.25775_103.9275,1.25775,103.9275,2026-05-14,1.50,Case Density
1,290337,1.25775_103.9275,1.25775,103.9275,2026-04-19,1.35,Case Density
2,290768,1.25775_103.9275,1.25775,103.9275,2026-04-25,1.35,Case Density
3,293726,1.25775_103.9275,1.25775,103.9275,2026-04-14,1.26,Case Density
4,298480,1.25775_103.9275,1.25775,103.9275,2026-01-14,0.00,Case Density


## Step 5: Configure Gemini API

In [6]:
client_ai = genai.Client(api_key="YOUR_API_KEY")

In [7]:
sample = inspection_df.iloc[0]

In [10]:
sample = inspection_df.iloc[0]

prompt = f"""
You are a dengue control advisor.

Location: {sample['cell_id']}

Risk Score: {sample['risk_score']}

Top Risk Factor: {sample['top_factor']}

Based on this information:

1. Explain why this area is risky.
2. Suggest three inspection actions.
3. Mention whether inspection should be High, Medium or Low priority.

Keep your answer under 80 words.
"""

In [11]:
response = client_ai.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)

print(response.text)

This area is risky due to high Case Density (Risk Score 1.5), indicating recent dengue cases.

**Inspection actions:**
1. Focus on residential homes, common areas, and construction sites.
2. Check water-holding containers, drains, and plant pot plates thoroughly.
3. Engage residents to 'Search and Destroy' breeding sites.

**Priority:** High


## Step 6: Generate AI Recommendations

In [12]:
def generate_recommendation(row):

    prompt = f"""
    You are an experienced dengue public health officer.

    Location:
    {row['cell_id']}

    Risk Score:
    {row['risk_score']}

    Top Risk Factor:
    {row['top_factor']}

    Explain:
    1. Why this area is risky.
    2. Three preventive actions.
    3. Inspection priority.

    Keep it below 80 words.
    """

    response = client_ai.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [13]:
inspection_df["AI_Recommendation"] = inspection_df.apply(
    generate_recommendation,
    axis=1
)

In [14]:
inspection_df.head()

,rank,cell_id,lat,lon,date,risk_score,top_factor,AI_Recommendation
0,276176,1.25775_103.9275,1.25775,103.9275,2026-05-14,1.50,Case Density,This area's high risk (1.5) stems from dense d...
1,290337,1.25775_103.9275,1.25775,103.9275,2026-04-19,1.35,Case Density,"This area (1.257N, 103.927E) is highly risky (..."
2,290768,1.25775_103.9275,1.25775,103.9275,2026-04-25,1.35,Case Density,This area (1.25775_103.9275) is high risk (1.3...
3,293726,1.25775_103.9275,1.25775,103.9275,2026-04-14,1.26,Case Density,"This area (1.25775_103.9275, Risk 1.26) is hig..."
4,298480,1.25775_103.9275,1.25775,103.9275,2026-01-14,0.00,Case Density,This area (1.25775_103.9275) shows no current ...


In [15]:
inspection_df.to_csv(
    "inspection_with_ai.csv",
    index=False
)

print("File saved successfully!")

File saved successfully!


## Step 7: Export Recommendations to CSV

In [17]:
inspection_df.to_csv("inspection_with_ai.csv", index=False)

print("CSV saved successfully!")

CSV saved successfully!


In [18]:
inspection_df["AI_Recommendation"].isnull().sum()

np.int64(0)

In [19]:
inspection_df[["risk_score","top_factor","AI_Recommendation"]].head(5)

,risk_score,top_factor,AI_Recommendation
0,1.50,Case Density,This area's high risk (1.5) stems from dense d...
1,1.35,Case Density,"This area (1.257N, 103.927E) is highly risky (..."
2,1.35,Case Density,This area (1.25775_103.9275) is high risk (1.3...
3,1.26,Case Density,"This area (1.25775_103.9275, Risk 1.26) is hig..."
4,0.00,Case Density,This area (1.25775_103.9275) shows no current ...


In [20]:
inspection_df.to_csv("inspection_with_ai.csv", index=False)

In [21]:
import os

os.listdir()

['.config', 'inspection_with_ai.csv', 'sample_data']

In [23]:
query = """
SELECT *
FROM `dengue-early-warning.dengue_ew.inspection_priority`
"""

In [24]:
inspection_df = client.query(query).to_dataframe()

# Conclusion

The Gemini AI module successfully converts dengue risk scores into actionable recommendations for health officials. These recommendations can be integrated into the dashboard to support faster and better decision-making during dengue outbreaks.